相较v1版本代码的改进点：

通过对特征与标签的相关性检验，筛选出最具相关性的特征


In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

train = pd.read_csv('/Kaggle-competition/Diabetes/train_diabetes.csv')
test = pd.read_csv('/Kaggle-competition/Diabetes/test_diabetes.csv')

In [15]:
# 检验特征与标签的相关性，并排序
corr_series = train.corr()['Diabetes'].sort_values(key=abs, ascending=False)
# 选择相关性绝对值大于0.1的特征
selected_features = corr_series[abs(corr_series) > 0.1].index.tolist()
selected_features.remove('Diabetes')  # 去掉目标变量本身

selected_features

['Age', 'ExerciseHours']

根据筛选，Age和ExerciseHours与Diabetes的相关性最强

接下来就以这两个为特征进行逻辑回归模型训练

In [21]:
train['ExerciseHours'].isna().sum()
train['Age'].isna().sum()

test['ExerciseHours'].isna().sum()
test['ExerciseHours'].isna().sum()
# 检验以后，均无空值

np.int64(0)

In [22]:
features = ['ExerciseHours', 'Age']
X_train = train[features]
y_train = train[('Diabetes')]
X_test = test[features]

model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

output = pd.DataFrame({'PatientID':test['PatientID'],'Diabetes': predictions})
output.to_csv('/Users/caierchang/Desktop/Diabetes_prediction_v2.csv', index=False)

关键发现

v2 vs v1

准确率下降：70.5% → 64.0%（下降6.5%）

精确率微升：71.43% → 71.91%（轻微提升）

召回率大幅下降：70.0% → 53.0%（严重下降17%）

F1分数下降：70.71% → 60.92%（下降9.8%）

问题分析

漏诊严重：47%的糖尿病患者被漏诊（假阴性太多）

过度保守：你的模型变得太谨慎，预测"健康"的比例增加

修改方向：你修改了69个预测，但只有15个改对了，28个反而改错了

具体表现

健康人识别：特异性从71%提高到75%（稍微更好）

患者识别：召回率从70%降到53%（大幅变差）

净效果：减少了误诊，但增加了漏诊，总体效果变差

